In [ ]:
import sys

print(sys.executable)
print(sys.version)

In [ ]:
import sys
!"{sys.executable}" -m pip install xgboost

In [ ]:
import numpy as np
import pandas as pd
import sklearn
import xgboost as xgb
import matplotlib

print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)
print("Scikit-learn:", sklearn.__version__)
print("XGBoost:", xgb.__version__)
print("Matplotlib:", matplotlib.__version__)
print("✓ Libraries OK")

In [ ]:
# ============================================================
# NOTEBOOK 08A — TABULAR MODELS
# Persistence + Ridge + Random Forest + XGBoost
# Uses the FROZEN 07D protocol.
#
# Primary comparison:
# - exactly the same node-level samples
# - 15 sensor inputs = 5 timestamps x [T, RH, CO2]
# - target: temperature at t+15 min
# - no calendar features
# - no test-set model selection
# ============================================================

from pathlib import Path
import json, time, platform, sys
import numpy as np
import pandas as pd

from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# XGBoost is optional at import time only to provide a clear error.
try:
    import xgboost as xgb
except ImportError as e:
    raise ImportError(
        "xgboost is not installed in this Python environment. "
        "Install it once with: pip install xgboost"
    ) from e

ROOT = Path.cwd().resolve()
IN = ROOT / "outputs" / "07D_final_alignment"
OUT = ROOT / "outputs" / "08A_tabular_models"
OUT.mkdir(parents=True, exist_ok=True)

UNSCALED = IN / "02_tabular_core_15_unscaled.csv"
SCALED = IN / "03_tabular_core_15_scaled.csv"
PROTOCOL = IN / "11_FROZEN_EXPERIMENTAL_PROTOCOL.json"

for p in [UNSCALED, SCALED, PROTOCOL]:
    assert p.exists(), f"Missing required 07D file: {p}"

TARGET = "target_temperature_t_plus_15"
META = ["uuid", "timestamp", "target_timestamp", "split", "segment_id", TARGET]

EXPECTED = {"train": 64701, "validation": 13825, "test": 13790}
SEED = 42

print("=" * 72)
print("NOTEBOOK 08A — PERSISTENCE + RIDGE + RANDOM FOREST + XGBOOST")
print("=" * 72)
print("Input:", IN)
print("Output:", OUT)

# ------------------------------------------------------------
# 1. LOAD FROZEN DATA
# ------------------------------------------------------------
du = pd.read_csv(UNSCALED)
ds = pd.read_csv(SCALED)

for d in [du, ds]:
    d["timestamp"] = pd.to_datetime(d["timestamp"], utc=True)
    d["target_timestamp"] = pd.to_datetime(d["target_timestamp"], utc=True)

features = [c for c in du.columns if c not in META]
assert len(features) == 15, f"Expected 15 primary features, got {len(features)}"
assert features == [c for c in ds.columns if c not in META]

# Exact row/key/target identity between scaled and unscaled files
assert len(du) == len(ds)
assert du[["uuid","timestamp","split"]].equals(ds[["uuid","timestamp","split"]])
assert np.allclose(du[TARGET], ds[TARGET], rtol=0, atol=1e-8)

counts = du["split"].value_counts().reindex(["train","validation","test"]).to_dict()
assert counts == EXPECTED, f"Frozen counts changed: {counts}"

print("\nFrozen counts:", counts)
print("Primary features:", len(features))

# ------------------------------------------------------------
# 2. SPLIT ARRAYS
# ------------------------------------------------------------
def arrays(frame, split):
    part = frame[frame["split"] == split].copy()
    return part, part[features].to_numpy(np.float32), part[TARGET].to_numpy(np.float32)

train_u, Xtr_u, ytr = arrays(du, "train")
val_u, Xva_u, yva = arrays(du, "validation")
test_u, Xte_u, yte = arrays(du, "test")

train_s, Xtr_s, ytr_s = arrays(ds, "train")
val_s, Xva_s, yva_s = arrays(ds, "validation")
test_s, Xte_s, yte_s = arrays(ds, "test")

assert np.allclose(ytr, ytr_s)
assert np.allclose(yva, yva_s)
assert np.allclose(yte, yte_s)

# Current temperature is the persistence predictor.
assert "temperature_t" in features
current_temp_idx = features.index("temperature_t")

# ------------------------------------------------------------
# 3. METRICS
# ------------------------------------------------------------
def metrics(y, pred):
    err = pred - y
    return {
        "MAE": float(mean_absolute_error(y, pred)),
        "RMSE": float(np.sqrt(mean_squared_error(y, pred))),
        "R2": float(r2_score(y, pred)),
        "Bias": float(np.mean(err)),
        "Error_SD": float(np.std(err, ddof=1)),
    }

results = []
pred_frames = []

def record(model_name, split, meta, y, pred, train_seconds=np.nan,
           selected_hyperparameters=""):
    m = metrics(y, pred)
    results.append({
        "model": model_name,
        "split": split,
        "n": len(y),
        **m,
        "train_seconds": train_seconds,
        "selected_hyperparameters": selected_hyperparameters,
    })
    p = meta[["uuid","timestamp","target_timestamp","split"]].copy()
    p["y_true"] = y
    p["y_pred"] = pred
    p["error"] = pred - y
    p["model"] = model_name
    pred_frames.append(p)

# ------------------------------------------------------------
# 4. PERSISTENCE BASELINE
# ------------------------------------------------------------
# y_hat(t+15) = T(t)
print("\n[1/4] Persistence")
for split, meta, X, y in [
    ("train", train_u, Xtr_u, ytr),
    ("validation", val_u, Xva_u, yva),
    ("test", test_u, Xte_u, yte),
]:
    pred = X[:, current_temp_idx]
    record("Persistence", split, meta, y, pred)

# ------------------------------------------------------------
# 5. RIDGE — alpha selected ONLY on validation
# ------------------------------------------------------------
print("[2/4] Ridge")
ridge_alphas = [0.01, 0.1, 1.0, 10.0, 100.0]
ridge_search = []

for alpha in ridge_alphas:
    model = Ridge(alpha=alpha)
    model.fit(Xtr_s, ytr)
    pred_val = model.predict(Xva_s)
    ridge_search.append({
        "alpha": alpha,
        "validation_RMSE": metrics(yva, pred_val)["RMSE"]
    })

ridge_search = pd.DataFrame(ridge_search).sort_values("validation_RMSE")
ridge_search.to_csv(OUT / "01_ridge_validation_search.csv", index=False)
best_alpha = float(ridge_search.iloc[0]["alpha"])

# Freeze selected alpha; fit TRAIN only for primary evaluation.
t0 = time.perf_counter()
ridge = Ridge(alpha=best_alpha)
ridge.fit(Xtr_s, ytr)
ridge_time = time.perf_counter() - t0

for split, meta, X, y in [
    ("train", train_s, Xtr_s, ytr),
    ("validation", val_s, Xva_s, yva),
    ("test", test_s, Xte_s, yte),
]:
    record("Ridge", split, meta, y, ridge.predict(X), ridge_time,
           json.dumps({"alpha": best_alpha}))

# ------------------------------------------------------------
# 6. RANDOM FOREST — compact predeclared validation grid
# ------------------------------------------------------------
print("[3/4] Random Forest")
rf_candidates = [
    {"n_estimators": 300, "max_depth": None, "min_samples_leaf": 1, "max_features": 1.0},
    {"n_estimators": 300, "max_depth": 20,   "min_samples_leaf": 2, "max_features": 1.0},
    {"n_estimators": 300, "max_depth": 12,   "min_samples_leaf": 2, "max_features": 1.0},
    {"n_estimators": 300, "max_depth": 20,   "min_samples_leaf": 4, "max_features": 0.8},
]

rf_search = []
for params in rf_candidates:
    model = RandomForestRegressor(
        **params, random_state=SEED, n_jobs=-1
    )
    model.fit(Xtr_u, ytr)
    pred_val = model.predict(Xva_u)
    rf_search.append({
        **params,
        "validation_RMSE": metrics(yva, pred_val)["RMSE"]
    })

rf_search = pd.DataFrame(rf_search).sort_values("validation_RMSE")
rf_search.to_csv(OUT / "02_rf_validation_search.csv", index=False)

best_rf = rf_search.iloc[0].to_dict()
best_rf.pop("validation_RMSE")
# Restore integer/None types safely
for k in ["n_estimators", "min_samples_leaf"]:
    best_rf[k] = int(best_rf[k])
if pd.notna(best_rf["max_depth"]):
    best_rf["max_depth"] = int(best_rf["max_depth"])
else:
    best_rf["max_depth"] = None

t0 = time.perf_counter()
rf = RandomForestRegressor(**best_rf, random_state=SEED, n_jobs=-1)
rf.fit(Xtr_u, ytr)
rf_time = time.perf_counter() - t0

for split, meta, X, y in [
    ("train", train_u, Xtr_u, ytr),
    ("validation", val_u, Xva_u, yva),
    ("test", test_u, Xte_u, yte),
]:
    record("RandomForest", split, meta, y, rf.predict(X), rf_time,
           json.dumps(best_rf))

# ------------------------------------------------------------
# 7. XGBOOST — compact predeclared validation grid
# ------------------------------------------------------------
print("[4/4] XGBoost")
xgb_candidates = [
    {"n_estimators": 300, "max_depth": 3, "learning_rate": 0.05,
     "subsample": 0.9, "colsample_bytree": 0.9},
    {"n_estimators": 500, "max_depth": 3, "learning_rate": 0.03,
     "subsample": 0.9, "colsample_bytree": 0.9},
    {"n_estimators": 300, "max_depth": 5, "learning_rate": 0.05,
     "subsample": 0.9, "colsample_bytree": 0.9},
    {"n_estimators": 500, "max_depth": 5, "learning_rate": 0.03,
     "subsample": 0.9, "colsample_bytree": 0.9},
]

xgb_search = []
for params in xgb_candidates:
    model = xgb.XGBRegressor(
        **params,
        objective="reg:squarederror",
        random_state=SEED,
        n_jobs=-1,
        tree_method="hist",
        reg_lambda=1.0,
        verbosity=0,
    )
    model.fit(Xtr_u, ytr)
    pred_val = model.predict(Xva_u)
    xgb_search.append({
        **params,
        "validation_RMSE": metrics(yva, pred_val)["RMSE"]
    })

xgb_search = pd.DataFrame(xgb_search).sort_values("validation_RMSE")
xgb_search.to_csv(OUT / "03_xgb_validation_search.csv", index=False)

best_xgb = xgb_search.iloc[0].to_dict()
best_xgb.pop("validation_RMSE")
for k in ["n_estimators", "max_depth"]:
    best_xgb[k] = int(best_xgb[k])
for k in ["learning_rate", "subsample", "colsample_bytree"]:
    best_xgb[k] = float(best_xgb[k])

t0 = time.perf_counter()
xgbr = xgb.XGBRegressor(
    **best_xgb,
    objective="reg:squarederror",
    random_state=SEED,
    n_jobs=-1,
    tree_method="hist",
    reg_lambda=1.0,
    verbosity=0,
)
xgbr.fit(Xtr_u, ytr)
xgb_time = time.perf_counter() - t0

for split, meta, X, y in [
    ("train", train_u, Xtr_u, ytr),
    ("validation", val_u, Xva_u, yva),
    ("test", test_u, Xte_u, yte),
]:
    record("XGBoost", split, meta, y, xgbr.predict(X), xgb_time,
           json.dumps(best_xgb))

# ------------------------------------------------------------
# 8. SAVE RESULTS + PAIRED TEST PREDICTIONS
# ------------------------------------------------------------
results_df = pd.DataFrame(results)
results_df.to_csv(OUT / "04_metrics_all_splits.csv", index=False)

pred_df = pd.concat(pred_frames, ignore_index=True)
pred_df.to_csv(OUT / "05_predictions_all_models.csv", index=False)

test_metrics = (
    results_df[results_df["split"] == "test"]
    .sort_values("RMSE")
    .reset_index(drop=True)
)
test_metrics.to_csv(OUT / "06_TEST_metrics_primary.csv", index=False)

# Per-device TEST metrics
per_device = []
test_predictions = pred_df[pred_df["split"] == "test"]
for (model, uuid), g in test_predictions.groupby(["model", "uuid"]):
    mm = metrics(g["y_true"].to_numpy(), g["y_pred"].to_numpy())
    per_device.append({"model": model, "uuid": uuid, "n": len(g), **mm})
pd.DataFrame(per_device).to_csv(
    OUT / "07_TEST_metrics_per_device.csv", index=False
)

# Model improvement over persistence on TEST
pers_rmse = float(
    test_metrics.loc[test_metrics["model"] == "Persistence", "RMSE"].iloc[0]
)
improvement = test_metrics[["model","RMSE","MAE","R2","Bias"]].copy()
improvement["RMSE_improvement_vs_persistence_pct"] = (
    100 * (pers_rmse - improvement["RMSE"]) / pers_rmse
)
improvement.to_csv(
    OUT / "08_TEST_improvement_vs_persistence.csv", index=False
)

# Feature importance / coefficients for interpretability
pd.DataFrame({
    "feature": features,
    "ridge_coefficient": ridge.coef_
}).to_csv(OUT / "09_ridge_coefficients.csv", index=False)

pd.DataFrame({
    "feature": features,
    "rf_importance": rf.feature_importances_
}).sort_values("rf_importance", ascending=False).to_csv(
    OUT / "10_rf_feature_importance.csv", index=False
)

pd.DataFrame({
    "feature": features,
    "xgb_importance": xgbr.feature_importances_
}).sort_values("xgb_importance", ascending=False).to_csv(
    OUT / "11_xgb_feature_importance.csv", index=False
)

# Reproducibility manifest
manifest = {
    "stage": "08A",
    "frozen_input": str(IN),
    "models": ["Persistence", "Ridge", "RandomForest", "XGBoost"],
    "primary_features": features,
    "n_primary_features": len(features),
    "target": TARGET,
    "forecast_horizon_minutes": 15,
    "sample_counts": EXPECTED,
    "hyperparameter_selection": "validation RMSE only",
    "test_used_for_selection": False,
    "ridge_scaled": True,
    "rf_scaled": False,
    "xgboost_scaled": False,
    "random_seed": SEED,
    "python": sys.version,
    "platform": platform.platform(),
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "sklearn": __import__("sklearn").__version__,
    "xgboost": xgb.__version__,
}
with (OUT / "12_reproducibility_manifest.json").open("w", encoding="utf-8") as f:
    json.dump(manifest, f, indent=2)

print("\n" + "=" * 72)
print("SELECTED HYPERPARAMETERS")
print("=" * 72)
print("Ridge:", {"alpha": best_alpha})
print("RF   :", best_rf)
print("XGB  :", best_xgb)

print("\n" + "=" * 72)
print("PRIMARY TEST RESULTS")
print("=" * 72)
print(
    test_metrics[
        ["model","n","MAE","RMSE","R2","Bias","Error_SD"]
    ].to_string(index=False)
)

print("\n" + "=" * 72)
print("IMPROVEMENT VS PERSISTENCE — TEST")
print("=" * 72)
print(
    improvement[
        ["model","RMSE","RMSE_improvement_vs_persistence_pct"]
    ].to_string(index=False)
)

print("\nVALIDATION")
print("✓ Frozen 07D samples used")
print("✓ 15-feature primary tabular representation only")
print("✓ Persistence = T(t)")
print("✓ Hyperparameters selected on validation only")
print("✓ Test never used for model selection")
print("✓ Ridge uses train-fitted scaling")
print("✓ RF/XGBoost use unscaled physical features")
print("✓ Same TEST observations for every 08A model")
print("✓ Per-device test metrics saved")
print("✓ Predictions saved for later paired statistical analysis")
